In [1]:
import os
try:
    path_initialized
except NameError:
    path_initialized = True
    os.chdir('..')

import numpy as np
from sympy.abc import x, y

from qldpc import codes
import networkx as nx

import src.device as device
from src.RotatedSurfaceCode import RotatedSurfaceCode

In [10]:
ll = [1,2,3,4,5,6]

list(zip(ll, ll[1:]))[::2]

[(1, 2), (3, 4), (5, 6)]

In [2]:
d = 5
dev = device.UnitCellDevice(d+2, d+2, device.hardware_params(1e-3))
code = RotatedSurfaceCode(d)

data_coords = {q:((code.qubit_coords[q][0]+1)//2, (code.qubit_coords[q][1]+1)//2) for q in code.data_indices}
assert len(set(data_coords.values())) == len(data_coords)
sched = dev.compile_SE_schedule_greedy(
    code,
    data_coords,
    code.check_cx_layers,
    use_highways=True,
    refocus_shuttle_noise=False,
    debug_qubit=48
)

25 (1, 3) [(1, 3), (1, 2)]
POP (1, 3, 'READOUT', 0) with cost 1600 (0 + 1600)
	ADD (1, 3, 'INTERACTION_ZONE', 0) FROM (1, 3, 'READOUT', 0) WITH COST 100
		estimated cost: 1600
	ADD (1, 3, 'SHUTTLE_INTERSECTION', 0) FROM (1, 3, 'READOUT', 0) WITH COST 100
		estimated cost: 1700
POP (1, 3, 'INTERACTION_ZONE', 0) with cost 1600 (100 + 1500)
	ADD (1, 3, 'READOUT', 0) FROM (1, 3, 'INTERACTION_ZONE', 0) WITH COST 100
		talready seen, cost 0, new cost 200
		pass
	ADD (1, 3, 'SHUTTLE_INTERSECTION', 0) FROM (1, 3, 'INTERACTION_ZONE', 0) WITH COST 100
		talready seen, cost 100, new cost 200
		pass
	ADD (1, 3, 'INTERACTION_ZONE', 1) FROM (1, 3, 'INTERACTION_ZONE', 0) WITH COST 100
		estimated cost: 1600
POP (1, 3, 'INTERACTION_ZONE', 1) with cost 1600 (200 + 1400)
	ADD (1, 3, 'READOUT', 1) FROM (1, 3, 'INTERACTION_ZONE', 1) WITH COST 100
		estimated cost: 1700
	ADD (1, 3, 'SHUTTLE_INTERSECTION', 1) FROM (1, 3, 'INTERACTION_ZONE', 1) WITH COST 100
		estimated cost: 1600
POP (1, 3, 'SHUTTLE_INTERSE

In [4]:
# construct the second-to-last code in Table 3 of arXiv:2308.07915v2, with code parameters [n, k, d] = [360, 12, <=24]
orders = {x: 30, y: 6}
poly_a = x**9 + y + y**2
poly_b = y**3 + x**25 + x**26
code = codes.BBCode(orders, poly_a, poly_b)

print(code)
print()
print("number of logical qubits:", code.dimension)

# find an upper bound to the code distance with 100 Monte Carlo trials
# 100 trials is likely not enough to reach the upper bound of 24 found by IBM
print("code distance: <=", code.get_distance_bound(num_trials=100))

BBCode on 360 qubits with cyclic group orders {x: 30, y: 6} and generating polynomials
  A = x**9 + y**2 + y
  B = x**26 + x**25 + y**3

number of logical qubits: 12
code distance: <= 56


In [1]:
import stim

In [4]:
circ = stim.Circuit()
circ.append('R', [0, 1])
circ.append('H', [0])
circ.append('CNOT', [0, 1])
circ.append('M', [0, 1])
circ.append('DETECTOR', stim.target_rec(-1), (0.0, 0.0, 0.0))
print(circ)

R 0 1
H 0
CX 0 1
M 0 1
DETECTOR(0, 0, 0) rec[-1]


In [6]:
for instr in circ:
    print(instr.name)

R
H
CX
M
DETECTOR
